Bert

In [1]:
import pandas as pd

# Load dataset
data = pd.read_csv('/content/Stress.csv')

# Preprocess text data (Convert to lowercase)
data['text'] = data['text'].astype(str).apply(lambda x: x.lower())

# Check class distribution
print("Class Distribution Before Balancing:")
print(data['label'].value_counts())  # Check the number of samples per class


Class Distribution Before Balancing:
label
0    1350
2    1011
1     477
Name: count, dtype: int64


In [2]:
# Handle class imbalance (Oversampling the minority classes)
class_counts = data['label'].value_counts()
max_count = class_counts.max()

balanced_data = []
for label in data['label'].unique():
    subset = data[data['label'] == label]
    balanced_data.append(subset.sample(max_count, replace=True))  # Oversample minority classes

data_balanced = pd.concat(balanced_data)

# Verify class distribution after balancing
print("Class Distribution After Balancing:")
print(data_balanced['label'].value_counts())

# Extract text and labels again
X = data_balanced['text']
y = data_balanced['label']


Class Distribution After Balancing:
label
2    1350
0    1350
1    1350
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

# Split dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [4]:
from transformers import BertTokenizer

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the input text
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
import torch

# Define dataset class
class StressDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values)  # Convert labels to tensor

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Create dataset objects for training and testing
train_dataset = StressDataset(train_encodings, y_train)
test_dataset = StressDataset(test_encodings, y_test)


In [6]:
from transformers import BertForSequenceClassification

# Load BERT model for sequence classification with 3 output labels
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',  # Directory to save model outputs
    num_train_epochs=20,  # Train for 20 epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    # evaluation_strategy="epoch",  # Evaluate at every epoch - Removed due to TypeError in older transformers versions
    # save_strategy="epoch",  # Save model after every epoch - Removed for consistency with evaluation_strategy
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
from transformers import Trainer

# Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


Step,Training Loss
500,0.882117
1000,0.499739
1500,0.252048
2000,0.101792
2500,0.040609
3000,0.032449
3500,0.018410
4000,0.035027
4500,0.017541
5000,0.003782


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8100, training_loss=0.11812433083116272, metrics={'train_runtime': 2276.1948, 'train_samples_per_second': 28.469, 'train_steps_per_second': 3.559, 'total_flos': 4262437367193600.0, 'train_loss': 0.11812433083116272, 'epoch': 20.0})

In [14]:
from sklearn.metrics import classification_report

# Evaluate the model on test data
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, pred_labels, target_names=['Low', 'Moderate', 'High']))



Classification Report:
              precision    recall  f1-score   support

         Low       0.94      0.86      0.90       270
    Moderate       0.87      0.94      0.90       270
        High       0.90      0.90      0.90       270

    accuracy                           0.90       810
   macro avg       0.90      0.90      0.90       810
weighted avg       0.90      0.90      0.90       810



In [16]:
def classify_stress_level(user_input):
    inputs = tokenizer(user_input, return_tensors='pt', truncation=True, padding=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax().item()
    stress_levels = ['Low', 'Moderate', 'High']
    return stress_levels[predicted_class]


In [18]:
def classify_stress_level(user_input):
    inputs = tokenizer(user_input, return_tensors='pt', truncation=True, padding=True, max_length=128)

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move tensors to the device where the model is located

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax().item()
    stress_levels = ['Low', 'Moderate', 'High']
    return stress_levels[predicted_class]

In [ ]:
while True:
    user_input = input("Enter a sentence to classify stress level (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        break
    stress_level = classify_stress_level(user_input)
    print(f"Predicted Stress Level: {stress_level}")


Enter a sentence to classify stress level (or type 'exit' to quit): Im so happy today !
Predicted Stress Level: Low
Enter a sentence to classify stress level (or type 'exit' to quit): u r a mother fucker
Predicted Stress Level: High
Enter a sentence to classify stress level (or type 'exit' to quit): i got 100000 rs today
Predicted Stress Level: Low
